# OPV Physics-Informed Gaussian Process Regression

This notebook compares a standard Gaussian Process Regression (GPR) model with a family of physics-informed GPR models for predicting organic photovoltaic (OPV) power conversion efficiency (PCE).

The physics-informed models modify the GP mean function. The GP kernel still learns residual structure from the data, but the prior mean encodes OPV mechanisms suggested by the paper: frontier-orbital near-degeneracy, exciton binding and dissociation, and charge transport.

## Paper Reference

This example uses the dataset associated with:

H. Sahu, W. Rao, A. Troisi, and H. Ma, **"Toward Predicting Efficiency of Organic Solar Cells via Machine Learning and Improved Descriptors,"** *Advanced Energy Materials*, 8(24), 1801032, 2018. DOI: [10.1002/aenm.201801032](https://doi.org/10.1002/aenm.201801032).

The key physical message used here is that PCE depends on multiple microscopic processes, and the paper highlights the importance of orbitals energetically close to HOMO and LUMO. In particular, small donor `Delta_H`, donor `Delta_L`, and acceptor `Delta_LA` can allow orbitals beyond HOMO/LUMO to participate in exciton formation, exciton dissociation, and charge transport.

## Physics-Informed Mean Functions Tested

The learning curve compares four models with increasing prior-mean complexity:

| Model | Mean function information | Physical motivation |
| --- | --- | --- |
| Standard GPR | learned constant mean | data-only baseline |
| PI-GPR 1 | frontier-orbital near-degeneracy | smaller `delHD`, `delLD`, and `delLA` should help OPV processes |
| PI-GPR 2 | near-degeneracy + exciton binding | lower `E_bind` should favor charge separation |
| PI-GPR 3 | near-degeneracy + exciton binding + transport | larger conjugation/polarizability and lower hole reorganization energy should improve transport |

The most useful comparison is usually at small training-set sizes. Physics-informed means should help most when the GP has limited data and needs a sensible prior trend.

## 1. Setup

The package is still local to this repository. This setup cell finds the local package whether the notebook is run from the workspace-level `examples/opv` folder or from an `examples/opv` folder inside the package repository.

In [ ]:
from pathlib import Path
import os
import sys

# Helpful on local macOS scientific Python stacks that load multiple OpenMP runtimes.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    """Find the local package root containing pyproject.toml and genmatics_gpr."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "genmatics_gpr").exists():
            return candidate
        sibling = candidate / "genmatics-gpr"
        if (sibling / "pyproject.toml").exists() and (sibling / "genmatics_gpr").exists():
            return sibling
    raise FileNotFoundError("Could not find the local genmatics_gpr package root.")


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Using package from: {PROJECT_ROOT}")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from genmatics_gpr import (
    PhysicsInformedMean,
    drop_duplicate_rows,
    fit_gpytorch_gpr,
    normalize_column_names,
    plot_correlation_matrix,
    plot_distribution,
    plot_parity,
    regression_metrics,
    replace_missing_placeholders,
    summarize_missingness,
    summarize_numeric_columns,
)

RANDOM_STATE = 42
plt.style.use("default")
torch.set_num_threads(1)
warnings.filterwarnings("ignore", category=RuntimeWarning)

## 2. Load and Clean the Dataset

The original file uses names such as `AL-DH` and `DL-AL`. The workflow keeps a descriptor table with the original names, then normalizes dataframe columns so the modeling code can use stable Python-friendly names.

In [ ]:
def find_data_path(start: Path) -> Path:
    """Find dataset.csv in the current OPV example folder."""
    candidates = [start / "dataset.csv"]
    candidates += [parent / "examples" / "opv" / "dataset.csv" for parent in start.parents]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find examples/opv/dataset.csv")


DATA_PATH = find_data_path(NOTEBOOK_DIR)
raw_df = pd.read_csv(DATA_PATH)
print(f"Loaded {DATA_PATH}")
raw_df.head()

In [ ]:
descriptor_info = pd.DataFrame(
    [
        ("polarizability", "polarizability", "Polarizability of donor molecules"),
        ("delLA", "della", "LUMO/LUMO+1 energetic difference of acceptors"),
        ("delLD", "delld", "LUMO/LUMO+1 energetic difference of donors"),
        ("N_atom", "n_atom", "Number of unsaturated atoms in the donor main conjugation path"),
        ("Eg", "eg", "Strongest singlet excited-state transition energy"),
        ("lamda_h", "lamda_h", "Hole reorganization energy in donor molecules"),
        ("DIP", "dip", "Vertical ionization potential of donor molecules"),
        ("AL-DH", "al_dh", "Donor HOMO / acceptor LUMO energetic difference"),
        ("delHD", "delhd", "HOMO/HOMO-1 energetic difference of donors"),
        ("E_bind", "e_bind", "Hole-electron binding energy in donor molecules"),
        ("DL-AL", "dl_al", "Donor LUMO / acceptor LUMO energetic difference"),
        ("delGE", "delge", "Dipole-moment change from ground state to first excited state"),
        ("E_T1", "e_t1", "Lowest triplet excited-state transition energy"),
    ],
    columns=["paper_column", "model_column", "description"],
)

descriptor_info

In [ ]:
df = (
    raw_df
    .pipe(replace_missing_placeholders)
    .pipe(normalize_column_names)
    .pipe(drop_duplicate_rows)
)

feature_columns = descriptor_info["model_column"].tolist()
target_column = "pce"

missing_columns = [column for column in feature_columns + [target_column] if column not in df.columns]
if missing_columns:
    raise KeyError(f"Missing expected columns after cleaning: {missing_columns}")

model_df = df[[target_column] + feature_columns].copy()
print(f"Rows: {model_df.shape[0]}")
print(f"Features: {len(feature_columns)}")
summarize_missingness(model_df)

In [ ]:
summarize_numeric_columns(model_df).round(3)

## 3. Quick Data Checks

These plots are not used for fitting, but they help confirm that the target and descriptor space look reasonable before building physics-informed models.

In [ ]:
fig, ax = plot_distribution(
    model_df[target_column],
    bins=20,
    title="Distribution of OPV Power Conversion Efficiency",
    xlabel="PCE (%)",
)
plt.show()

In [ ]:
fig, ax, corr = plot_correlation_matrix(
    model_df,
    columns=[target_column] + feature_columns,
    figsize=(10, 8),
    title="Correlation Matrix for OPV Descriptors and PCE",
    annotate=False,
)
plt.show()

In [ ]:
correlation_with_pce = (
    model_df.corr(numeric_only=True)[target_column]
    .drop(target_column)
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
correlation_with_pce.to_frame("pearson_r_with_pce").round(3)

## 4. Physics-Informed Mean Functions

Each equation returns a mean PCE in the original target units. The GP then learns residual deviations around that mean. The equations use standardized physical scores so the learnable weights have a simple interpretation: positive weights reward physically favorable descriptors.

In [ ]:
def zscore_feature(features, parameters, name):
    """Return a standardized feature tensor using training-set statistics."""
    return (features[name] - parameters[f"{name}_mean"]) / parameters[f"{name}_std"]


def frontier_degeneracy_score(features, parameters):
    """Higher score means smaller frontier-orbital gaps, which the paper links to higher PCE."""
    donor_h = -zscore_feature(features, parameters, "delhd")
    donor_l = -zscore_feature(features, parameters, "delld")
    acceptor_l = -zscore_feature(features, parameters, "della")
    return (donor_h + donor_l + acceptor_l) / 3.0


def exciton_separation_score(features, parameters):
    """Higher score means lower hole-electron binding energy."""
    return -zscore_feature(features, parameters, "e_bind")


def transport_score(features, parameters):
    """Higher score means larger conjugation/polarizability and lower hole reorganization energy."""
    log_n_atom = torch.log(torch.clamp(features["n_atom"], min=1.0))
    log_polarizability = torch.log(torch.clamp(features["polarizability"], min=1.0))
    conjugation = (log_n_atom - parameters["log_n_atom_mean"]) / parameters["log_n_atom_std"]
    polarizability = (
        (log_polarizability - parameters["log_polarizability_mean"])
        / parameters["log_polarizability_std"]
    )
    reorganization = -zscore_feature(features, parameters, "lamda_h")
    return (conjugation + polarizability + reorganization) / 3.0


def degeneracy_mean_equation(features, parameters):
    return (
        parameters["baseline"]
        + parameters["degeneracy_weight"] * frontier_degeneracy_score(features, parameters)
    )


def degeneracy_binding_mean_equation(features, parameters):
    return (
        parameters["baseline"]
        + parameters["degeneracy_weight"] * frontier_degeneracy_score(features, parameters)
        + parameters["binding_weight"] * exciton_separation_score(features, parameters)
    )


def degeneracy_binding_transport_mean_equation(features, parameters):
    return (
        parameters["baseline"]
        + parameters["degeneracy_weight"] * frontier_degeneracy_score(features, parameters)
        + parameters["binding_weight"] * exciton_separation_score(features, parameters)
        + parameters["transport_weight"] * transport_score(features, parameters)
    )

In [ ]:
PHYSICS_MODEL_SPECS = {
    "standard_gpr": {
        "label": "Standard GPR",
        "equation": None,
        "features": [],
        "weights": [],
        "description": "Constant mean baseline with a learned GP covariance.",
    },
    "pi_gpr_degeneracy": {
        "label": "PI-GPR 1: degeneracy",
        "equation": degeneracy_mean_equation,
        "features": ["delhd", "delld", "della"],
        "weights": ["degeneracy_weight"],
        "description": "Rewards small donor/acceptor frontier-orbital gaps.",
    },
    "pi_gpr_degeneracy_binding": {
        "label": "PI-GPR 2: degeneracy + binding",
        "equation": degeneracy_binding_mean_equation,
        "features": ["delhd", "delld", "della", "e_bind"],
        "weights": ["degeneracy_weight", "binding_weight"],
        "description": "Adds low exciton binding energy as a favorable term.",
    },
    "pi_gpr_degeneracy_binding_transport": {
        "label": "PI-GPR 3: degeneracy + binding + transport",
        "equation": degeneracy_binding_transport_mean_equation,
        "features": ["delhd", "delld", "della", "e_bind", "n_atom", "polarizability", "lamda_h"],
        "weights": ["degeneracy_weight", "binding_weight", "transport_weight"],
        "description": "Adds conjugation, polarizability, and hole reorganization energy.",
    },
}

pd.DataFrame(
    [
        {
            "model_key": key,
            "label": spec["label"],
            "physics_features": ", ".join(spec["features"]) or "none",
            "description": spec["description"],
        }
        for key, spec in PHYSICS_MODEL_SPECS.items()
    ]
)

In [ ]:
def physics_statistics(X_train):
    """Training-set statistics used inside the physics equations."""
    stats = {}
    for column in ["delhd", "delld", "della", "e_bind", "lamda_h"]:
        std = float(X_train[column].std(ddof=0))
        stats[f"{column}_mean"] = float(X_train[column].mean())
        stats[f"{column}_std"] = std if std > 0 else 1.0

    log_n_atom = np.log(np.clip(X_train["n_atom"].to_numpy(), 1.0, None))
    log_polarizability = np.log(np.clip(X_train["polarizability"].to_numpy(), 1.0, None))
    stats["log_n_atom_mean"] = float(log_n_atom.mean())
    stats["log_n_atom_std"] = float(log_n_atom.std(ddof=0) or 1.0)
    stats["log_polarizability_mean"] = float(log_polarizability.mean())
    stats["log_polarizability_std"] = float(log_polarizability.std(ddof=0) or 1.0)
    return stats


def build_physics_mean(model_key, X_train, y_train, feature_scaler):
    """Create the requested PhysicsInformedMean for scaled GPyTorch inputs."""
    spec = PHYSICS_MODEL_SPECS[model_key]
    if spec["equation"] is None:
        return None

    learnable_parameters = {"baseline": float(np.mean(y_train))}
    learnable_parameters.update({weight: 0.5 for weight in spec["weights"]})

    feature_indices = {column: feature_columns.index(column) for column in spec["features"]}
    feature_means = {
        column: float(feature_scaler.mean_[feature_columns.index(column)])
        for column in spec["features"]
    }
    feature_stds = {
        column: float(feature_scaler.scale_[feature_columns.index(column)])
        for column in spec["features"]
    }

    return PhysicsInformedMean(
        equation=spec["equation"],
        feature_indices=feature_indices,
        learnable_parameters=learnable_parameters,
        positive_parameters=tuple(spec["weights"]),
        fixed_parameters=physics_statistics(X_train),
        feature_means=feature_means,
        feature_stds=feature_stds,
    )

## 5. Holdout Split

The learning curves use a fixed holdout test set. Training subsets are sampled from the remaining pool with approximate stratification over PCE, following the paper's motivation to keep the full efficiency range represented.

In [ ]:
X = model_df[feature_columns].copy()
y = model_df[target_column].copy()

X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Training pool rows: {len(X_train_pool)}")
print(f"Holdout test rows: {len(X_test)}")

In [ ]:
def stratified_fraction_indices(y_values, fraction, random_state, n_bins=5):
    """Sample a fraction of rows while preserving the target distribution approximately."""
    if fraction >= 1.0:
        return y_values.index

    bins = pd.qcut(y_values, q=n_bins, labels=False, duplicates="drop")
    sample_frame = pd.DataFrame({"bin": bins}, index=y_values.index)
    sampled = sample_frame.groupby("bin", group_keys=False).sample(
        frac=fraction,
        random_state=random_state,
    )
    return sampled.index


def root_mean_squared_error(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def pearson_r(y_true, y_pred):
    if len(y_true) < 2:
        return np.nan
    return float(pearsonr(np.asarray(y_true).ravel(), np.asarray(y_pred).ravel())[0])

## 6. Learning Curve Experiment

The default settings are intentionally modest so the notebook runs quickly on a laptop. Increase `N_REPEATS` and `TRAINING_ITER` for a smoother figure when preparing a final publication-style example.

In [ ]:
TRAIN_FRACTIONS = [0.10, 0.20, 0.40, 0.60, 1.00]
N_REPEATS = 2
TRAINING_ITER = 80
LEARNING_RATE = 0.05
KERNEL = "matern"
MODEL_ORDER = list(PHYSICS_MODEL_SPECS)

print(f"Models: {[PHYSICS_MODEL_SPECS[key]['label'] for key in MODEL_ORDER]}")
print(f"Train fractions: {TRAIN_FRACTIONS}")
print(f"Repeats per fraction: {N_REPEATS}")
print(f"GPR optimization iterations per fit: {TRAINING_ITER}")

In [ ]:
def fit_and_score_gp(model_key, X_train_subset, y_train_subset, X_eval, y_eval):
    """Fit one GPyTorch GPR model and score it on the holdout set."""
    scaler = StandardScaler().fit(X_train_subset)
    X_train_scaled = scaler.transform(X_train_subset).copy()
    X_eval_scaled = scaler.transform(X_eval).copy()

    mean_module = build_physics_mean(
        model_key,
        X_train_subset,
        y_train_subset,
        scaler,
    )

    result = fit_gpytorch_gpr(
        X_train_scaled,
        y_train_subset.to_numpy(),
        kernel=KERNEL,
        ard=True,
        mean_module=mean_module,
        lr=LEARNING_RATE,
        training_iter=TRAINING_ITER,
        initial_noise=0.1,
        standardize_y=True,
        verbose=False,
    )

    prediction = result.predict(X_eval_scaled, return_std=True)
    metrics = regression_metrics(y_eval, prediction.mean, prefix="test")
    metrics["test_RMSE"] = root_mean_squared_error(y_eval, prediction.mean)
    metrics["test_r"] = pearson_r(y_eval, prediction.mean)
    return result, prediction, metrics

In [ ]:
learning_curve_records = []
final_fit_cache = {}

for fraction in TRAIN_FRACTIONS:
    for repeat in range(N_REPEATS):
        subset_seed = RANDOM_STATE + repeat + int(fraction * 1000)
        subset_index = stratified_fraction_indices(
            y_train_pool,
            fraction=fraction,
            random_state=subset_seed,
        )
        X_subset = X_train_pool.loc[subset_index]
        y_subset = y_train_pool.loc[subset_index]

        for model_key in MODEL_ORDER:
            result, prediction, metrics = fit_and_score_gp(
                model_key,
                X_subset,
                y_subset,
                X_test,
                y_test,
            )
            record = {
                "model_key": model_key,
                "model": PHYSICS_MODEL_SPECS[model_key]["label"],
                "train_fraction": fraction,
                "train_size": len(X_subset),
                "repeat": repeat,
                **metrics,
            }
            learning_curve_records.append(record)

            if fraction == 1.0 and repeat == 0:
                final_fit_cache[model_key] = {
                    "result": result,
                    "prediction": prediction,
                    "train_size": len(X_subset),
                }

learning_curve = pd.DataFrame(learning_curve_records)
learning_curve.head()

In [ ]:
learning_summary = (
    learning_curve
    .groupby(["model_key", "model", "train_fraction", "train_size"], as_index=False)
    .agg(
        test_RMSE_mean=("test_RMSE", "mean"),
        test_RMSE_std=("test_RMSE", "std"),
        test_r_mean=("test_r", "mean"),
        test_r_std=("test_r", "std"),
        test_R2_mean=("test_R2", "mean"),
        test_R2_std=("test_R2", "std"),
    )
)

learning_summary[[
    "model",
    "train_size",
    "test_RMSE_mean",
    "test_RMSE_std",
    "test_r_mean",
    "test_r_std",
    "test_R2_mean",
]].round(3)

## 7. Learning Curve Plots

The left panel shows prediction error, where lower is better. The right panel shows Pearson correlation with holdout PCE, where higher is better. The shaded bands show one standard deviation across repeated stratified training subsets.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
colors = {
    "standard_gpr": "tab:gray",
    "pi_gpr_degeneracy": "tab:blue",
    "pi_gpr_degeneracy_binding": "tab:green",
    "pi_gpr_degeneracy_binding_transport": "tab:red",
}
markers = {
    "standard_gpr": "o",
    "pi_gpr_degeneracy": "s",
    "pi_gpr_degeneracy_binding": "^",
    "pi_gpr_degeneracy_binding_transport": "D",
}

for model_key in MODEL_ORDER:
    model_data = learning_summary[learning_summary["model_key"] == model_key].sort_values("train_size")
    label = PHYSICS_MODEL_SPECS[model_key]["label"]
    color = colors[model_key]
    marker = markers[model_key]

    axes[0].plot(
        model_data["train_size"],
        model_data["test_RMSE_mean"],
        marker=marker,
        linewidth=2,
        color=color,
        label=label,
    )
    axes[0].fill_between(
        model_data["train_size"],
        model_data["test_RMSE_mean"] - model_data["test_RMSE_std"].fillna(0.0),
        model_data["test_RMSE_mean"] + model_data["test_RMSE_std"].fillna(0.0),
        color=color,
        alpha=0.12,
    )

    axes[1].plot(
        model_data["train_size"],
        model_data["test_r_mean"],
        marker=marker,
        linewidth=2,
        color=color,
        label=label,
    )
    axes[1].fill_between(
        model_data["train_size"],
        model_data["test_r_mean"] - model_data["test_r_std"].fillna(0.0),
        model_data["test_r_mean"] + model_data["test_r_std"].fillna(0.0),
        color=color,
        alpha=0.12,
    )

axes[0].set_title("Learning Curve: Holdout RMSE")
axes[0].set_xlabel("Training samples")
axes[0].set_ylabel("RMSE in PCE (%)")
axes[0].grid(True, alpha=0.25)

axes[1].set_title("Learning Curve: Holdout Pearson r")
axes[1].set_xlabel("Training samples")
axes[1].set_ylabel("Pearson r")
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(True, alpha=0.25)
axes[1].legend(frameon=False, loc="lower right")

fig.suptitle("Standard GPR vs Physics-Informed GPR on OPV PCE", y=1.03)
fig.tight_layout()
plt.show()

## 8. Final Holdout Parity Plot

This compares the standard model with the most complex physics-informed mean using the full training pool. If the physics prior is useful, its predictions should move closer to the ideal diagonal, especially for high- and low-PCE regions where small datasets are hardest.

In [ ]:
standard_prediction = final_fit_cache["standard_gpr"]["prediction"]
physics_prediction = final_fit_cache["pi_gpr_degeneracy_binding_transport"]["prediction"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

for ax, title, prediction in [
    (axes[0], "Standard GPR", standard_prediction),
    (axes[1], "PI-GPR 3: degeneracy + binding + transport", physics_prediction),
]:
    metrics = regression_metrics(y_test, prediction.mean)
    min_value = min(y_test.min(), prediction.mean.min()) - 0.5
    max_value = max(y_test.max(), prediction.mean.max()) + 0.5
    ax.errorbar(
        y_test,
        prediction.mean,
        yerr=prediction.std,
        fmt="o",
        markersize=6,
        alpha=0.85,
        capsize=2,
        markeredgecolor="black",
        markeredgewidth=0.4,
    )
    ax.plot([min_value, max_value], [min_value, max_value], "--", color="black", linewidth=1.2)
    ax.set_xlim(min_value, max_value)
    ax.set_ylim(min_value, max_value)
    ax.set_title(title)
    ax.set_xlabel("Experimental PCE (%)")
    ax.grid(True, alpha=0.25)
    ax.text(
        0.05,
        0.95,
        f"RMSE = {root_mean_squared_error(y_test, prediction.mean):.2f}\n"
        f"r = {metrics['r']:.2f}\n"
        f"R2 = {metrics['R2']:.2f}",
        transform=ax.transAxes,
        va="top",
        bbox={"boxstyle": "round", "facecolor": "white", "edgecolor": "gray", "alpha": 0.9},
    )

axes[0].set_ylabel("Predicted PCE (%)")
fig.tight_layout()
plt.show()

## 9. Learned Physics Mean Parameters

The learned positive weights show how strongly the model uses each physics prior term after joint optimization with the GP covariance. They are not causal coefficients, but they are useful diagnostics for whether the prior mean is being used.

In [ ]:
parameter_rows = []
for model_key, cached in final_fit_cache.items():
    mean_module = cached["result"].model.mean_module
    if hasattr(mean_module, "current_parameter_values"):
        values = mean_module.current_parameter_values()
        parameter_rows.extend(
            {
                "model": PHYSICS_MODEL_SPECS[model_key]["label"],
                "parameter": name,
                "value": value,
            }
            for name, value in values.items()
        )

pd.DataFrame(parameter_rows).round(4)

## 10. Interpretation

For this OPV dataset, the most defensible physics-informed mean is the frontier-orbital near-degeneracy term because it is the paper's main mechanistic insight. Adding exciton binding energy and transport proxies increases complexity while staying chemically interpretable.

Expected behavior:

- Physics-informed GPR should help most at small training sizes.
- The standard GPR may catch up as the training set grows because the kernel sees enough data to learn the trend directly.
- If a more complex physics mean underperforms a simpler one, that is useful information: not every plausible physical proxy improves the prior mean for this dataset.

For a final public example, rerun with more repeats and optimization iterations, then save the learning-curve table alongside the notebook.